# Evaluación con Tuning — Prophet (alineado con SARIMAX / Árboles)

**Prophet es el primo directo de SARIMAX en tu comparativo:** no usa lags ni rolling, recibe el
calendario como **regresores / festivos**, y predice el mes completo de una sola vez. Por eso este
notebook replica el molde del de SARIMAX para que la comparación sea justa.

- Splits por fecha idénticos: `TRAIN_START='2018-01'`, `TEST_START='2024-09'`, test = todos los meses hasta el final.
- Folds de tuning = `N_CV_FOLDS` meses previos al test (walk-forward CV mensual).
- Métricas idénticas: **MAE, RMSE, MAPE (%), R²**.
- Todo el análisis agregado (promedios, comparaciones, conclusiones) se hace **solo sobre meses estables**;
  los atípicos se marcan con ⚠ y se excluyen de los promedios.

**Qué usa de tu CSV (según la tabla del README para Prophet: contexto ✓, lags ✗, rolling ✗, cíclicas ✗):**
- `Consumo` → `y`, índice de fecha → `ds`.
- Festivos colombianos como mecanismo de *holidays* de Prophet (construidos desde la columna `es_festivo`
  del propio CSV, con ventana ±1 día para capturar `dia_antes/despues_festivo`).
- Solo regresores de tipo **evento** (`es_mantenimiento`, `covid_cuarentena`, `semana_navidad`).
  Los ordinales de calendario (`dia_semana`, `mes`, `es_fin_semana`) **no** se añaden: son redundantes con
  las estacionalidades semanal/anual que Prophet deriva de la fecha.


## 1. Imports

In [ ]:
import warnings, time, json, itertools, os, contextlib, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pandas.tseries.offsets import MonthBegin, MonthEnd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/EAFIT/SIMAT/Metro/data/

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.facecolor': 'white'})

# Prophet (pip install prophet)
try:
    from prophet import Prophet
    HAS_PROPHET = True
    print("\u2713 Prophet disponible")
except ImportError:
    HAS_PROPHET = False
    print("\u2717 Prophet no instalado \u2014 ejecutar: pip install prophet")

# Silenciar logs de Prophet / cmdstanpy
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

try:
    from tqdm.auto import tqdm; HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    class tqdm:
        def __init__(self, iterable=None, **kw): self._it = iterable or []
        def __iter__(self): return iter(self._it)
        def set_postfix(self, **kw): pass

print("\u2713 Librerías base listas")

## 2. Configuración global

**Ajusta aquí fechas, meses atípicos, festivos y parámetros del tuning.**

| Parámetro | Descripción |
|-----------|-------------|
| `TRAIN_START` / `TEST_START` | Mes inicial de train / test (alineado con SARIMAX) |
| `N_CV_FOLDS` | Meses de walk-forward CV (antes del test) para el tuning |
| `SAMPLE` / `N_SAMPLE` | Si `True`, evalúa solo `N_SAMPLE` combinaciones aleatorias de la grilla |
| `HOLIDAY_LOWER/UPPER` | Ventana (días) alrededor de cada festivo |
| `REGRESSORS` | Regresores de evento a añadir (deben ser conocidos a futuro) |
| `ATYPICAL_MONTHS` | Meses excluidos de todos los promedios |


In [ ]:
# ─── AJUSTA AQUÍ ──────────────────────────────────────────────────────────────
CSV_PATH    = 'features_mes_completo.csv'   # mismo CSV que SARIMAX / árboles

TRAIN_START = '2018-01'   # alineado con SARIMAX
TEST_START  = '2024-09'   # todo desde aquí = test

N_CV_FOLDS  = 6           # folds walk-forward (meses antes del test) para tuning
SAMPLE      = True        # muestreo aleatorio de la grilla (Prophet es rápido)
N_SAMPLE    = 40          # nº de combinaciones a evaluar si SAMPLE=True
SEED        = 42

# Festivos
USE_CSV_FESTIVOS    = True    # construir holidays desde la columna es_festivo del CSV
USE_COUNTRY_HOLIDAYS = False  # alternativa: m.add_country_holidays('CO')
HOLIDAY_LOWER = -1            # ventana inferior (captura 'dia_antes_festivo')
HOLIDAY_UPPER =  1            # ventana superior (captura 'dia_despues_festivo')

# Regresores de evento (conocidos a futuro). Se filtran automáticamente si no existen
# o si son constantes en el train de un fold.
REGRESSORS = ['es_mantenimiento', 'covid_cuarentena', 'semana_navidad']

# Meses atípicos -> EXCLUIDOS de todos los promedios / análisis agregado
ATYPICAL_MONTHS = ['2024-09', '2025-03', '2025-05', '2025-10', '2025-11', '2025-12']

# Mostrar también los meses atípicos en las gráficas por-mes (⚠). False = solo estables.
SHOW_ALL_MONTHS_IN_PLOTS = True
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(SEED)

COLORS = {'Prophet': '#00897B'}
MODELS = ['Prophet']

print(f"Modelos activos : {MODELS}")
print(f"Train desde     : {TRAIN_START}")
print(f"Test desde      : {TEST_START}  (hasta fin de datos)")
print(f"Folds CV tuning : {N_CV_FOLDS}")
print(f"Regresores      : {REGRESSORS}")
print(f"Festivos        : {'CSV es_festivo' if USE_CSV_FESTIVOS else ('CO country' if USE_COUNTRY_HOLIDAYS else 'ninguno')}"
      f"  (ventana {HOLIDAY_LOWER}..{HOLIDAY_UPPER})")
print(f"Meses atípicos  : {ATYPICAL_MONTHS}")

## 3. Carga del dataset

In [ ]:
df = pd.read_csv(CSV_PATH, index_col='Fecha', parse_dates=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)
df = df.asfreq('D')                       # frecuencia diaria de calendario
df = df[df.index >= pd.Timestamp(TRAIN_START + '-01')].copy()

# booleanos -> int
bool_cols = df.select_dtypes(include='bool').columns.tolist()
if bool_cols:
    df[bool_cols] = df[bool_cols].astype(int)

print(f"Shape           : {df.shape}")
print(f"Rango de fechas : {df.index.min().date()} \u2192 {df.index.max().date()}")
nulos = df.isnull().sum(); nc = nulos[nulos > 0]
print(f"Columnas con nulos: {len(nc)}" + (f"\n{nc}" if len(nc) else " \u2014 ninguno"))
print(f"\nConsumo \u2014 media: {df['Consumo'].mean():,.0f}  "
      f"std: {df['Consumo'].std():,.0f}  "
      f"min: {df['Consumo'].min():,.0f}  max: {df['Consumo'].max():,.0f}")

## 4. Festivos y regresores

Se construye el `DataFrame` de festivos que Prophet usará tanto al entrenar como al predecir
(cubre train + test + futuro), y se validan los regresores de evento contra las columnas del CSV.


In [ ]:
# ── Festivos para Prophet ─────────────────────────────────────────────────────
HOLIDAYS_DF = None
if USE_CSV_FESTIVOS and 'es_festivo' in df.columns:
    hol_dates = df.index[df['es_festivo'].astype(bool)]
    HOLIDAYS_DF = pd.DataFrame({
        'holiday'     : 'festivo_co',
        'ds'          : hol_dates,
        'lower_window': HOLIDAY_LOWER,
        'upper_window': HOLIDAY_UPPER,
    })
    print(f"\u2713 Festivos desde CSV: {len(HOLIDAYS_DF)} fechas marcadas (ventana {HOLIDAY_LOWER}..{HOLIDAY_UPPER})")
elif USE_COUNTRY_HOLIDAYS:
    print("\u2713 Se usarán festivos de Colombia vía add_country_holidays('CO')")
else:
    print("\u26a0 Sin festivos explícitos (solo estacionalidad semanal/anual)")

# ── Regresores disponibles ────────────────────────────────────────────────────
REGRESSORS = [r for r in REGRESSORS if r in df.columns]
TARGET = 'Consumo'
print(f"\u2713 Regresores disponibles: {REGRESSORS}")

## 5. Definición dinámica de splits

Idéntica a SARIMAX / árboles: train desde `TRAIN_START`, test desde `TEST_START` hasta el final,
folds CV = `N_CV_FOLDS` meses previos al test, y meses estables = test \\ atípicos.


In [ ]:
def month_row_count(period):
    start = pd.Timestamp(str(period) + '-01'); end = start + MonthEnd(1)
    return len(df[(df.index >= start) & (df.index <= end)])

test_start = pd.Timestamp(TEST_START + '-01')

p = test_start.to_period('M'); last_period = df.index.max().to_period('M')
TEST_MONTHS = []
while p <= last_period:
    if month_row_count(p) >= 20: TEST_MONTHS.append(str(p))
    p += 1
if not TEST_MONTHS:
    raise ValueError(f"No hay meses completos de test desde {TEST_START}.")

p = (test_start - MonthBegin(1)).to_period('M')
CV_MONTHS = []
while len(CV_MONTHS) < N_CV_FOLDS:
    if month_row_count(p) >= 20: CV_MONTHS.insert(0, str(p))
    p -= 1

STABLE_MONTHS   = [m for m in TEST_MONTHS if m not in ATYPICAL_MONTHS]
ATYP_IN_TEST    = [m for m in TEST_MONTHS if m in ATYPICAL_MONTHS]
ANALYSIS_MONTHS = STABLE_MONTHS
PLOT_MONTHS     = TEST_MONTHS if SHOW_ALL_MONTHS_IN_PLOTS else STABLE_MONTHS

pre_test = df[df.index < test_start]
print("=" * 64)
print(f"  Dataset completo : {df.index.min().date()} \u2192 {df.index.max().date()}")
print(f"  Train (pre-test) : {pre_test.index.min().date()} \u2192 {pre_test.index.max().date()}  ({len(pre_test)} días)")
print(f"  Folds CV ({N_CV_FOLDS} m.) : {CV_MONTHS[0]} \u2192 {CV_MONTHS[-1]}")
print(f"  Test  ({len(TEST_MONTHS)} m.) : {TEST_MONTHS[0]} \u2192 {TEST_MONTHS[-1]}")
print(f"  \u2192 Estables ({len(STABLE_MONTHS)}) : {STABLE_MONTHS}")
print(f"  \u2192 Atípicos ({len(ATYP_IN_TEST)}) : {ATYP_IN_TEST}")
print("=" * 64)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
df['Consumo'].plot(ax=ax, lw=0.7, color='#90A4AE', label='Train expandible')
for mes in CV_MONTHS:
    s = pd.Timestamp(f'{mes}-01'); e = s + MonthEnd(1)
    ax.axvspan(s, e, color='#FFA726', alpha=0.40)
ax.axvspan(0, 0, color='#FFA726', alpha=0.40, label=f'Folds CV tuning ({N_CV_FOLDS} meses)')
for mes in STABLE_MONTHS:
    s = pd.Timestamp(f'{mes}-01'); e = s + MonthEnd(1)
    ax.axvspan(s, e, color='#EF5350', alpha=0.45)
ax.axvspan(0, 0, color='#EF5350', alpha=0.45, label=f'Test estable ({len(STABLE_MONTHS)} meses)')
for mes in ATYP_IN_TEST:
    s = pd.Timestamp(f'{mes}-01'); e = s + MonthEnd(1)
    ax.axvspan(s, e, color='#6A1B9A', alpha=0.30, hatch='//')
ax.axvspan(0, 0, color='#6A1B9A', alpha=0.30, hatch='//',
           label=f'Test atípico ({len(ATYP_IN_TEST)} meses, excluidos de promedios)')
ax.set_title('Distribución Train / CV / Test  (atípicos excluidos de los promedios)', fontweight='bold')
ax.set_ylabel('Consumo'); ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.savefig('split_overview_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Funciones auxiliares

- `compute_metrics`: MAE, RMSE, MAPE, R² (idéntico a SARIMAX/árboles).
- `make_frame`: arma el DataFrame `ds`/`y`/regresores que espera Prophet.
- `fit_predict_prophet`: ajusta Prophet (festivos + regresores no-constantes) y predice el mes.
- `cv_score_prophet`: walk-forward CV mensual; retorna el MAPE medio (métrica de tuning).


In [ ]:
@contextlib.contextmanager
def _suppress_stan():
    """Silencia la salida C de cmdstanpy durante fit/predict."""
    with open(os.devnull, 'w') as devnull:
        old_out, old_err = os.dup(1), os.dup(2)
        try:
            os.dup2(devnull.fileno(), 1); os.dup2(devnull.fileno(), 2)
            yield
        finally:
            os.dup2(old_out, 1); os.dup2(old_err, 2)
            os.close(old_out); os.close(old_err)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    r2   = r2_score(y_true, y_pred)
    return {'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'MAPE (%)': round(mape, 4), 'R²': round(r2, 4)}


def make_frame(rows, regressors, with_y=True):
    """Construye el frame de Prophet (ds, [y], regresores) desde un slice de df."""
    f = pd.DataFrame({'ds': rows.index})
    if with_y:
        f['y'] = rows[TARGET].values
    for r in regressors:
        f[r] = rows[r].values
    return f


def fit_predict_prophet(params, hist_rows, fut_rows):
    """
    Ajusta Prophet en hist_rows y predice las fechas de fut_rows.
    Filtra regresores constantes en el train (Prophet falla con varianza cero).
    """
    eff_reg = [r for r in REGRESSORS if hist_rows[r].nunique() > 1]
    hist = make_frame(hist_rows, eff_reg, with_y=True)
    fut  = make_frame(fut_rows,  eff_reg, with_y=False)
    try:
        with _suppress_stan():
            m = Prophet(
                changepoint_prior_scale = params['changepoint_prior_scale'],
                seasonality_prior_scale = params['seasonality_prior_scale'],
                holidays_prior_scale    = params['holidays_prior_scale'],
                seasonality_mode        = params['seasonality_mode'],
                weekly_seasonality      = True,
                yearly_seasonality      = True,
                daily_seasonality       = False,
                holidays                = HOLIDAYS_DF,
            )
            if HOLIDAYS_DF is None and USE_COUNTRY_HOLIDAYS:
                m.add_country_holidays(country_name='CO')
            for r in eff_reg:
                m.add_regressor(r)
            m.fit(hist)
            fcst = m.predict(fut)
        return fcst['yhat'].values, m
    except Exception as e:
        print(f"    \u26a0 Error Prophet {params}: {e}")
        return np.full(len(fut_rows), np.nan), None


def cv_score_prophet(params, cv_months):
    """Walk-forward CV mensual dentro del train. Retorna MAPE medio."""
    mapes = []
    for mes in cv_months:
        cutoff = pd.Timestamp(f'{mes}-01'); end = cutoff + MonthEnd(1)
        tr  = df[df.index < cutoff]
        val = df[(df.index >= cutoff) & (df.index <= end)]
        if len(tr) < 365 or len(val) == 0:
            continue
        preds, _ = fit_predict_prophet(params, tr, val)
        if not np.all(np.isnan(preds)):
            mapes.append(compute_metrics(val[TARGET].values, preds)['MAPE (%)'])
        else:
            mapes.append(999.0)
    return float(np.mean(mapes)) if mapes else 999.0


print("\u2713 Funciones auxiliares definidas.")

## 7. Espacio de hiperparámetros y defaults

Se tunean los parámetros más influyentes de Prophet:

| Parámetro | Efecto |
|-----------|--------|
| `changepoint_prior_scale` | Flexibilidad de la tendencia (el más impactante) |
| `seasonality_prior_scale` | Fuerza de las estacionalidades |
| `seasonality_mode` | `additive` vs `multiplicative` (demanda eléctrica suele ser multiplicativa) |
| `holidays_prior_scale` | Fuerza del efecto de festivos |


In [ ]:
PARAM_GRID = {
    'changepoint_prior_scale': [0.01, 0.05, 0.10, 0.50],
    'seasonality_prior_scale': [1.0, 5.0, 10.0, 15.0],
    'seasonality_mode'       : ['additive', 'multiplicative'],
    'holidays_prior_scale'   : [1.0, 5.0, 10.0],
}
keys = list(PARAM_GRID.keys())
GRID = [dict(zip(keys, combo)) for combo in itertools.product(*PARAM_GRID.values())]

if SAMPLE and N_SAMPLE < len(GRID):
    idx = rng.choice(len(GRID), N_SAMPLE, replace=False)
    GRID = [GRID[i] for i in idx]
    print(f"Muestreo activo: {len(GRID)} combinaciones (de {int(np.prod([len(v) for v in PARAM_GRID.values()]))})")
else:
    print(f"Grilla completa: {len(GRID)} combinaciones")

DEFAULT_PARAMS = {
    'Prophet': {
        'changepoint_prior_scale': 0.05,    # default de Prophet
        'seasonality_prior_scale': 10.0,
        'seasonality_mode'       : 'additive',
        'holidays_prior_scale'   : 10.0,
    }
}
total_fits = len(GRID) * N_CV_FOLDS
print(f"Fits de CV estimados: {total_fits}  (Prophet entrena en ~1-3 s c/u)")
print("Parámetros por defecto:", DEFAULT_PARAMS['Prophet'])

## 8. Búsqueda de hiperparámetros (Grid Search + Walk-Forward CV)

Cada combinación se evalúa con walk-forward CV mensual sobre `CV_MONTHS` (todos previos a `TEST_START`,
por lo que el tuning ya se hace sobre comportamiento estable). Se elige la de menor **MAPE medio**.


In [ ]:
tuning_results = {}
best_params    = {}
t0_global = time.time()

for model_name in MODELS:
    print(f"\n{'\u2500'*60}\n  Tuning {model_name}\n{'\u2500'*60}")
    rows = []; t0 = time.time()
    for i, params in enumerate(tqdm(GRID, desc=model_name)):
        score = cv_score_prophet(params, CV_MONTHS)
        rows.append({'iter': i+1, **params, 'cv_mape': score})
        if (i+1) % 5 == 0 or (i+1) == len(GRID):
            best_so_far = min(r['cv_mape'] for r in rows)
            print(f"  combo {i+1:>3}/{len(GRID)}  |  MAPE actual={score:.3f}%  |  "
                  f"mejor={best_so_far:.3f}%  |  tiempo={time.time()-t0:.0f}s")
    df_res = pd.DataFrame(rows).sort_values('cv_mape').reset_index(drop=True)
    tuning_results[model_name] = df_res
    best_row = df_res.iloc[0]
    best_params[model_name] = {
        k: (str(best_row[k]) if k == 'seasonality_mode' else float(best_row[k]))
        for k in keys
    }
    print(f"\n  \u2713 Mejor MAPE CV = {best_row['cv_mape']:.3f}%")
    print(f"  Params: {best_params[model_name]}")

print(f"\n{'='*60}\n  Tuning completado en {(time.time()-t0_global)/60:.1f} min\n{'='*60}")

## 9. Resultados del tuning — Convergencia

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_res = tuning_results['Prophet'].sort_values('iter')
ax.scatter(df_res['iter'], df_res['cv_mape'], color=COLORS['Prophet'], alpha=0.6, s=40, zorder=3)
ax.plot(df_res['iter'], df_res['cv_mape'].cummin(), 'k--', lw=1.5, label='Mínimo acumulado')
best = df_res['cv_mape'].min()
ax.axhline(best, color='red', lw=1.2, ls=':', label=f'Mejor={best:.2f}%')
ax.set_title('Convergencia del Grid Search \u2014 Prophet', fontweight='bold')
ax.set_xlabel('Combinación evaluada'); ax.set_ylabel('MAPE CV (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.savefig('tuning_convergencia_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

print("\n\u2500\u2500 Mejores params \u2500\u2500")
for k, v in best_params['Prophet'].items():
    print(f"  {k:26s}: {v}")
print("\nTop 5 combinaciones:")
print(tuning_results['Prophet'].head(5).to_string(index=False))

## 10. Evaluación final en test

Para cada mes de test se re-entrena Prophet con **todo el train anterior** (ventana expandible) en dos
variantes: **Default** y **Tuneado**. Las exógenas (regresores/festivos) son conocidas a futuro,
así que no hay problema de disponibilidad como con los lags de los árboles.


In [ ]:
def eval_all_months(params_dict, label=''):
    res = {}
    for mes_test in TEST_MONTHS:
        cutoff = pd.Timestamp(f'{mes_test}-01'); end = cutoff + MonthEnd(1)
        tr = df[df.index < cutoff]
        te = df[(df.index >= cutoff) & (df.index <= end)]
        res[mes_test] = {}
        for model_name, params in params_dict.items():
            preds, model = fit_predict_prophet(params, tr, te)
            if np.all(np.isnan(preds)):
                metrics = {'MAE': np.nan, 'RMSE': np.nan, 'MAPE (%)': np.nan, 'R²': np.nan}
            else:
                metrics = compute_metrics(te[TARGET].values, preds)
            res[mes_test][model_name] = {
                'preds': preds, 'real': te[TARGET].values, 'dates': te.index,
                'metrics': metrics, 'model': model,
            }
        print(f"  \u2713 {mes_test} evaluado ({label})")
    return res

print("Evaluando con parámetros por defecto\u2026")
results_default = eval_all_months({m: DEFAULT_PARAMS[m] for m in MODELS}, label='default')
print("\nEvaluando con parámetros tuneados\u2026")
results_tuned   = eval_all_months(best_params, label='tuned')
print("\n\u2713 Evaluación completa.")

## 11. Predicción vs Real — parámetros tuneados

Todos los meses de `PLOT_MONTHS`; los atípicos van marcados con ⚠ y no entran en los promedios.


In [ ]:
def is_atyp(mes): return mes in ATYPICAL_MONTHS
n = len(PLOT_MONTHS)
fig, axes = plt.subplots(n, 1, figsize=(14, 4.5*n), sharex=False)
if n == 1: axes = [axes]
for ax, mes in zip(axes, PLOT_MONTHS):
    dates = results_tuned[mes]['Prophet']['dates']
    real  = results_tuned[mes]['Prophet']['real']
    ax.plot(dates, real, 'k-o', lw=2.2, ms=5, label='Real', zorder=6)
    p    = results_tuned[mes]['Prophet']['preds']
    mape = results_tuned[mes]['Prophet']['metrics']['MAPE (%)']
    lbl  = f"Prophet  MAPE={mape:.2f}%" if not np.isnan(mape) else "Prophet  error"
    ax.plot(dates, p, '--s', ms=4, lw=1.6, color=COLORS['Prophet'], alpha=0.9, label=lbl)
    tag = '  \u26a0 ATÍPICO (excluido de promedios)' if is_atyp(mes) else ''
    ax.set_title(f'Predicción vs Real (tuneado) \u2014 {mes}{tag}', fontsize=12, fontweight='bold',
                 color=('#6A1B9A' if is_atyp(mes) else 'black'))
    if is_atyp(mes): ax.set_facecolor('#F3E5F5')
    ax.set_ylabel('Consumo (kWh)'); ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.25, ls='--')
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%d-%b'))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.suptitle('Predicción vs Real \u2014 Prophet Tuneado', fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout(); plt.savefig('pred_vs_real_tuned_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Degradación del error con el horizonte — parámetros tuneados

In [ ]:
n = len(PLOT_MONTHS)
fig, axes = plt.subplots(n, 1, figsize=(14, 4.0*n))
if n == 1: axes = [axes]
for ax, mes in zip(axes, PLOT_MONTHS):
    real  = results_tuned[mes]['Prophet']['real']
    preds = results_tuned[mes]['Prophet']['preds']
    dias  = np.arange(1, len(real)+1)
    errors = np.abs(real - preds)
    ax.bar(dias, errors, width=0.6, color=COLORS['Prophet'], alpha=0.5, label='Prophet')
    trend = pd.Series(errors).rolling(5, center=True, min_periods=1).mean()
    ax.plot(dias, trend, '-', lw=2.2, color=COLORS['Prophet'])
    ax.axvline(7.5,  color='gray', lw=1.1, ls=':',  alpha=0.7, label='Día 7')
    ax.axvline(14.5, color='gray', lw=1.1, ls='--', alpha=0.5, label='Día 14')
    tag = '  \u26a0 ATÍPICO' if is_atyp(mes) else ''
    ax.set_title(f'Error Absoluto por Horizonte \u2014 {mes}{tag}', fontsize=12, fontweight='bold',
                 color=('#6A1B9A' if is_atyp(mes) else 'black'))
    ax.set_xlabel('Día dentro del mes'); ax.set_ylabel('|Error| (kWh)')
    ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.25, ls='--'); ax.set_xticks(dias)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.suptitle('Degradación del Error \u2014 Prophet Tuneado', fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout(); plt.savefig('error_horizonte_tuned_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 13. Tabla comparativa — Default vs Tuneado

Todos los meses con marca `Atípico`; el promedio destacado se calcula **solo sobre meses estables**.


In [ ]:
rows_def, rows_tun = [], []
for mes in TEST_MONTHS:
    for model_name in MODELS:
        for tag, res in [('Default', results_default), ('Tuned', results_tuned)]:
            m = res[mes][model_name]['metrics']
            (rows_def if tag == 'Default' else rows_tun).append({
                'Mes': mes, 'Modelo': model_name,
                'Atípico': 'Sí' if mes in ATYPICAL_MONTHS else 'No',
                'MAE': m['MAE'], 'RMSE': m['RMSE'], 'MAPE (%)': m['MAPE (%)'], 'R²': m['R²'],
            })
df_def = pd.DataFrame(rows_def).set_index(['Mes', 'Modelo'])
df_tun = pd.DataFrame(rows_tun).set_index(['Mes', 'Modelo'])
metric_cols = ['MAE', 'RMSE', 'MAPE (%)', 'R²']
df_delta = df_tun[metric_cols] - df_def[metric_cols]
df_delta.columns = [f'Δ {c}' for c in df_delta.columns]; df_delta['Atípico'] = df_def['Atípico']

print("\n\u2550\u2550 MÉTRICAS DEFAULT \u2550\u2550"); print(df_def.to_string())
print("\n\u2550\u2550 MÉTRICAS TUNEADAS \u2550\u2550"); print(df_tun.to_string())
print("\n\u2550\u2550 DELTA (Tuned \u2212 Default; negativo = mejora) \u2550\u2550"); print(df_delta.to_string())

def _avg(frame, months):
    sub = frame[frame.index.get_level_values('Mes').isin(months)]
    return sub.groupby('Modelo')[metric_cols].mean().round(2)

print("\n" + "\u2588"*64)
print(f"  PROMEDIO SOBRE MESES ESTABLES ({len(STABLE_MONTHS)} meses) \u2014 RESULTADO PRINCIPAL")
print("\u2588"*64)
print("\nDEFAULT (estables):"); print(_avg(df_def, STABLE_MONTHS).to_string())
print("\nTUNED (estables):");   print(_avg(df_tun, STABLE_MONTHS).to_string())
print("\nMejora media (Tuned \u2212 Default, estables):")
print((_avg(df_tun, STABLE_MONTHS) - _avg(df_def, STABLE_MONTHS)).round(2).to_string())
print("\n\u2500\u2500 (Referencia) Promedio incluyendo atípicos \u2500\u2500")
print("TUNED (todos):"); print(_avg(df_tun, TEST_MONTHS).to_string())

## 14. Comparativa visual — Default vs Tuneado (MAPE %, solo meses estables)

In [ ]:
avg_def = _avg(df_def, STABLE_MONTHS)['MAPE (%)']
avg_tun = _avg(df_tun, STABLE_MONTHS)['MAPE (%)']
x = np.arange(len(MODELS)); w = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - w/2, [avg_def.get(m, np.nan) for m in MODELS], w, label='Default',
            color='#90A4AE', edgecolor='white', alpha=0.9)
b2 = ax.bar(x + w/2, [avg_tun.get(m, np.nan) for m in MODELS], w, label='Tuneado',
            color=[COLORS[m] for m in MODELS], edgecolor='white', alpha=0.9)
for bars, bold in [(b1, False), (b2, True)]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x()+bar.get_width()/2, h+0.05, f'{h:.2f}%', ha='center',
                    va='bottom', fontsize=10, fontweight='bold' if bold else 'normal')
ax.set_xticks(x); ax.set_xticklabels(MODELS, fontsize=12)
ax.set_ylabel('MAPE promedio (%)')
ax.set_title(f'MAPE Promedio: Default vs Tuneado \u2014 {len(STABLE_MONTHS)} meses estables',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, axis='y', alpha=0.25, ls='--')
vals = [v for v in list(avg_def) + list(avg_tun) if not np.isnan(v)]
ax.set_ylim(0, max(vals)*1.25 if vals else 10)
plt.tight_layout(); plt.savefig('default_vs_tuned_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 15. Heatmap MAPE — parámetros tuneados (con fila 'Prom. ESTABLE')

In [ ]:
mape_series = df_tun['MAPE (%)'].xs('Prophet', level='Modelo').loc[TEST_MONTHS]
stable_avg  = mape_series.loc[STABLE_MONTHS].mean()
mat = np.array(list(mape_series.values) + [stable_avg]).reshape(-1, 1)
ylabels = [f'{m}  \u26a0' if m in ATYPICAL_MONTHS else m for m in mape_series.index] + ['Prom. ESTABLE']
fig, ax = plt.subplots(figsize=(4.2, max(4, len(ylabels)*0.5+1)))
vmin = np.nanmin(mat)*0.9; vmax = np.nanmax(mape_series.values)*1.1
im = ax.imshow(mat, cmap='RdYlGn_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_xticks([0]); ax.set_xticklabels(['Prophet'], fontsize=12, fontweight='bold')
ax.set_yticks(range(len(ylabels))); ax.set_yticklabels(ylabels, fontsize=10)
ax.axhline(len(mape_series.index)-0.5, color='black', lw=2)
mean_val = np.nanmean(mape_series.values)
for i in range(mat.shape[0]):
    v = mat[i, 0]
    ax.text(0, i, f'{v:.2f}%', ha='center', va='center', fontsize=11, fontweight='bold',
            color='white' if (not np.isnan(v) and v > mean_val) else 'black')
plt.colorbar(im, ax=ax, label='MAPE (%)')
ax.set_title('MAPE (%) \u2014 Prophet Tuneado', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('heatmap_mape_tuned_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 16. Componentes del modelo — Prophet tuneado

Análogo al *feature importance* de los árboles: Prophet descompone la serie en tendencia,
estacionalidades y efecto de festivos. Se ajusta sobre todo el pre-test con los mejores params.


In [ ]:
tr_full = df[df.index < test_start]
fut_full = make_frame(tr_full, [r for r in REGRESSORS if tr_full[r].nunique() > 1], with_y=False)
preds_full, m_full = fit_predict_prophet(best_params['Prophet'], tr_full, tr_full)
if m_full is not None:
    with _suppress_stan():
        fcst_full = m_full.predict(make_frame(tr_full,
                        [r for r in REGRESSORS if tr_full[r].nunique() > 1], with_y=False))
        fig = m_full.plot_components(fcst_full)
    fig.set_size_inches(11, 8)
    plt.tight_layout(); plt.savefig('componentes_prophet.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print("No fue posible ajustar el modelo para componentes.")

## 17. APE promedio por horizonte — meses estables

In [ ]:
records = []
for mes in STABLE_MONTHS:
    real = results_tuned[mes]['Prophet']['real']
    preds = results_tuned[mes]['Prophet']['preds']
    for i, (e, rv) in enumerate(zip(np.abs(real - preds), real)):
        records.append({'Horizonte': i+1, 'APE': e/rv*100 if rv != 0 else np.nan})
ape_df = pd.DataFrame(records)
ape_h  = ape_df.groupby('Horizonte')['APE'].mean()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ape_h.index, ape_h.values, '-o', ms=5, lw=2, color=COLORS['Prophet'], label='Prophet')
ax.axvline(7,  color='gray', lw=1.1, ls=':',  alpha=0.7, label='Día 7')
ax.axvline(14, color='gray', lw=1.1, ls='--', alpha=0.5, label='Día 14')
ax.set_xlabel('Horizonte (día del mes)'); ax.set_ylabel('APE promedio (%)')
ax.set_title('Degradación del Error por Horizonte \u2014 Prophet (meses estables)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.25); ax.set_xticks(ape_h.index)
plt.tight_layout(); plt.savefig('ape_horizonte_tuned_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 18. MAPE por mes — parámetros tuneados (⚠ = atípico, excluido de promedios)

In [ ]:
vals = [results_tuned[mes]['Prophet']['metrics']['MAPE (%)'] for mes in TEST_MONTHS]
x = np.arange(len(TEST_MONTHS))
fig, ax = plt.subplots(figsize=(max(11, len(TEST_MONTHS)*0.8), 5))
bars = ax.bar(x, vals, 0.6, color=COLORS['Prophet'], alpha=0.85, edgecolor='white')
for bar, mes in zip(bars, TEST_MONTHS):
    if mes in ATYPICAL_MONTHS: bar.set_hatch('//')
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x()+bar.get_width()/2, h+0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=8)
xt = [f'{m} \u26a0' if m in ATYPICAL_MONTHS else m for m in TEST_MONTHS]
ax.set_xticks(x); ax.set_xticklabels(xt, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('MAPE (%)')
ax.set_title('MAPE por Mes \u2014 Prophet Tuneado', fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.25, ls='--')
plt.tight_layout(); plt.savefig('mape_por_mes_prophet.png', dpi=150, bbox_inches='tight'); plt.show()

## 19. Resumen sobre meses estables y exportación

In [ ]:
d = _avg(df_def, STABLE_MONTHS).loc['Prophet']
t = _avg(df_tun, STABLE_MONTHS).loc['Prophet']
summary_stable = pd.DataFrame([{
    'Modelo': 'Prophet',
    'MAPE def': d['MAPE (%)'], 'MAPE tun': t['MAPE (%)'], 'Δ MAPE': round(t['MAPE (%)']-d['MAPE (%)'], 2),
    'MAE tun': t['MAE'], 'RMSE tun': t['RMSE'], 'R² tun': t['R²'],
}]).set_index('Modelo')
print("\u2550\u2550 RESUMEN \u2014 MESES ESTABLES \u2550\u2550")
print(summary_stable.to_string())
print(f"\n\u2192 Prophet (tuneado, meses estables): MAPE {t['MAPE (%)']:.2f}%  |  R² {t['R²']:.2f}")

all_rows = []
for mes in TEST_MONTHS:
    for tag, res in [('default', results_default), ('tuned', results_tuned)]:
        r = res[mes]['Prophet']
        for date, pred, real_v in zip(r['dates'], r['preds'], r['real']):
            pred = float(pred)
            all_rows.append({
                'Fecha': date.date(), 'Mes': mes, 'Atipico': mes in ATYPICAL_MONTHS,
                'Modelo': 'Prophet', 'Variante': tag, 'Real': real_v,
                'Prediccion': round(pred, 2) if not np.isnan(pred) else None,
                'AbsError': round(abs(real_v - pred), 2) if not np.isnan(pred) else None,
                'APE (%)': round(abs(real_v-pred)/real_v*100, 4) if (real_v != 0 and not np.isnan(pred)) else None,
            })
preds_df = pd.DataFrame(all_rows)
preds_df.to_csv('predicciones_tuned_prophet.csv', index=False)

df_def_exp = df_def.copy(); df_def_exp['variante'] = 'default'
df_tun_exp = df_tun.copy(); df_tun_exp['variante'] = 'tuned'
pd.concat([df_def_exp, df_tun_exp]).reset_index().to_csv('metricas_tuned_prophet.csv', index=False)
summary_stable.to_csv('resumen_meses_estables_prophet.csv')
with open('best_params_prophet.json', 'w') as f:
    json.dump(best_params, f, indent=2, default=str)

print("\nArchivos exportados:")
print("  predicciones_tuned_prophet.csv      \u2014 predicciones día a día (default + tuned, flag Atipico)")
print("  metricas_tuned_prophet.csv          \u2014 MAE/RMSE/MAPE/R² por mes (ambas variantes)")
print("  resumen_meses_estables_prophet.csv  \u2014 resumen promediado sobre meses estables")
print("  best_params_prophet.json            \u2014 mejores hiperparámetros")
print(f"\n{preds_df.head(8).to_string(index=False)}")